# RAG-Based Educational Chatbot
### With Retrieval + Reranking Pipeline

**Subjects Covered:** Mathematics | Science | English | Social Studies

---

## Task 1 — RAG Pipeline Design (with Reranking)

```
[PDF Study Materials]
        ↓
[Document Loading]            ← PyMuPDF extracts text from PDFs
        ↓
[Text Chunking]               ← Sliding window, 300 words, 50 overlap
        ↓
[Embedding Generation]        ← sentence-transformers → 384-dim vectors
        ↓
[ChromaDB Vector Storage]
        ↓
[Student Question]
        ↓
[Stage 1 — Retrieval]         ← top-30 candidates via cosine similarity
        ↓
[Stage 2 — Reranking]         ← cross-encoder scores all 30, keeps top-5
        ↓
[Prompt Engineering]          ← top-5 reranked chunks injected into prompt
        ↓
[OpenAI GPT-3.5-turbo]        ← Student-friendly answer
```

**Why two stages?**
- **Stage 1 (bi-encoder)** is fast but rough — retrieves 30 candidates using vector similarity
- **Stage 2 (cross-encoder reranker)** is slower but precise — reads question + chunk together and gives a relevance score
- Result: much more accurate context → much better answers

## Install Dependencies

In [ ]:
!pip install pymupdf chromadb sentence-transformers openai

## Task 2 — Chunking Strategy

**Strategy: Sliding Window with Overlap**

| Parameter | Value | Reason |
|-----------|-------|--------|
| Chunk size | 300 words | Covers one full concept/paragraph |
| Overlap | 50 words | Preserves context at boundaries |

Larger initial retrieval (top-30) means even loosely related chunks are candidates — the reranker then filters them down to the truly relevant ones.

In [ ]:
# Task 2 — Chunking Function

def chunk_text(text, chunk_size=300, overlap=50):
    """
    Splits text into overlapping word-based chunks.
    chunk_size : number of words per chunk
    overlap    : number of words shared between consecutive chunks
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunk = " ".join(words[start : start + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

print("Chunking function ready.")

## Task 3 — Embedding Model & Vector Database

**Bi-Encoder (for retrieval): `all-MiniLM-L6-v2`**
- Encodes question and chunks *separately* into vectors
- Very fast — used to scan all chunks and retrieve top-30 candidates
- Trade-off: doesn't compare question and chunk together, so less precise

**Cross-Encoder (for reranking): `cross-encoder/ms-marco-MiniLM-L-6-v2`**
- Takes *both* question and chunk as a pair and outputs a relevance score
- Much more accurate — understands the relationship between question and text
- Slower, so only used on the top-30 candidates from Stage 1

**Vector Database: ChromaDB** — in-memory, zero setup, cosine similarity

In [ ]:
# Task 3 — Load Bi-Encoder, Cross-Encoder, and ChromaDB

import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

# Bi-encoder: fast retrieval (Stage 1)
print("Loading bi-encoder embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Bi-encoder ready!")

# Cross-encoder: accurate reranking (Stage 2)
print("Loading cross-encoder reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder reranker ready!")

# Vector store
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="educational_docs",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB collection ready!")

## Task 4 — Prompt Engineering

| Prompt Element | Purpose |
|---|---|
| Role: "helpful tutor" | Sets a friendly, educational tone |
| "Use ONLY the context below" | Grounds answer in textbook — prevents hallucination |
| "If not in context, say so" | Honest fallback instead of making things up |
| "Simple language" | Suitable for school students |
| "Step by step if needed" | Helps explain math/science processes clearly |

Now using **top-5 reranked chunks** instead of raw retrieved chunks — much higher quality context fed to the LLM.

In [ ]:
# Task 4 — Prompt Template

def build_prompt(context_chunks, student_question):
    context = "\n\n".join(context_chunks)
    prompt = f"""You are a helpful and friendly tutor for school students.
Answer the student's question using ONLY the context provided below.
- Use simple, easy-to-understand language.
- Explain step by step if the question involves a process or calculation.
- If the answer is not in the context, say: "I'm not sure about that — please check your textbook."
- Do NOT make up any information.

--- CONTEXT FROM TEXTBOOK ---
{context}
-----------------------------

Student's Question: {student_question}

Answer:"""
    return prompt

print("Prompt template ready.")

## Task 5 — Build the Chatbot

### Step 5a: Upload Your PDF Files

Create a `pdfs/` folder next to this notebook and place your textbook PDFs inside:
```
pdfs/
  science.pdf
  mathematics.pdf
  english.pdf
  social_studies.pdf
```

In [ ]:
# Step 5a — Load and Extract Text from PDFs

import fitz  # PyMuPDF
import os

PDF_FOLDER = "pdfs"

def load_pdfs_from_folder(folder_path):
    """
    Reads all PDF files from the folder.
    Returns dict: { subject_name : extracted_text }
    """
    documents = {}

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created '{folder_path}/' folder. Add your PDF textbooks there and re-run.")
        return documents

    pdf_files = [f for f in os.listdir(folder_path) if f.endswith(".pdf")]

    if not pdf_files:
        print(f"No PDFs found in '{folder_path}/'. Please add your textbook PDFs.")
        return documents

    for filename in pdf_files:
        filepath = os.path.join(folder_path, filename)
        subject_name = os.path.splitext(filename)[0]
        doc = fitz.open(filepath)
        full_text = "".join(page.get_text() for page in doc)
        doc.close()
        documents[subject_name] = full_text
        print(f"  Loaded: {filename}  ({len(full_text.split())} words)")

    return documents


print("Loading PDFs...")
educational_documents = load_pdfs_from_folder(PDF_FOLDER)
print(f"\nSubjects loaded: {list(educational_documents.keys())}")

### Step 5b: Chunk, Embed, and Store in Vector DB

In [ ]:
# Step 5b — Chunk all PDFs, embed, and store in ChromaDB

all_chunks = []
all_ids = []
all_metadata = []

for subject, text in educational_documents.items():
    chunks = chunk_text(text, chunk_size=300, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{subject}_chunk_{i}")
        all_metadata.append({"subject": subject})
    print(f"  {subject}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print("Generating embeddings...")
embeddings = embedding_model.encode(all_chunks, batch_size=64, show_progress_bar=True).tolist()

collection.add(
    documents=all_chunks,
    embeddings=embeddings,
    ids=all_ids,
    metadatas=all_metadata
)
print("\nAll chunks stored in ChromaDB!")

### Step 5c: Stage 1 — Retrieve Top-30 Candidates

In [ ]:
# Step 5c — Stage 1: Retrieve top-30 candidates using bi-encoder (fast)

def retrieve_candidates(question, top_k=30):
    """
    Embeds the question and retrieves top_k most similar chunks
    from ChromaDB using cosine similarity (bi-encoder).
    Returns list of (chunk_text, subject) tuples.
    """
    question_embedding = embedding_model.encode([question]).tolist()

    results = collection.query(
        query_embeddings=question_embedding,
        n_results=top_k
    )

    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]

    candidates = [(chunk, meta["subject"]) for chunk, meta in zip(chunks, metadatas)]
    return candidates


# Test Stage 1
test_q = "What is photosynthesis?"
candidates = retrieve_candidates(test_q, top_k=30)
print(f"Stage 1 retrieved {len(candidates)} candidates for: '{test_q}'")
print(f"\nFirst candidate preview:\n{candidates[0][0][:200]}...")

### Step 5d: Stage 2 — Rerank with Cross-Encoder

In [ ]:
# Step 5d — Stage 2: Rerank the 30 candidates using cross-encoder (accurate)

def rerank_chunks(question, candidates, top_n=5):
    """
    Takes the question + all candidate chunks,
    scores each pair using the cross-encoder,
    and returns the top_n highest-scoring chunks.

    Cross-encoder reads question + chunk TOGETHER
    → much more accurate relevance score than bi-encoder alone.
    """
    chunk_texts = [c[0] for c in candidates]

    # Create (question, chunk) pairs for cross-encoder
    pairs = [(question, chunk) for chunk in chunk_texts]

    # Score all pairs
    scores = reranker.predict(pairs)

    # Sort by score descending and pick top_n
    scored_chunks = sorted(
        zip(scores, chunk_texts),
        key=lambda x: x[0],
        reverse=True
    )

    top_chunks = [chunk for score, chunk in scored_chunks[:top_n]]
    top_scores = [round(float(score), 4) for score, chunk in scored_chunks[:top_n]]

    return top_chunks, top_scores


# Test Stage 2
top_chunks, top_scores = rerank_chunks(test_q, candidates, top_n=5)
print(f"Stage 2 reranked to top-5 chunks for: '{test_q}'\n")
for i, (chunk, score) in enumerate(zip(top_chunks, top_scores), 1):
    print(f"[Rank {i} | Score: {score}]\n{chunk[:200]}...\n{'-'*55}")

### Step 5e: Set OpenAI API Key

In [ ]:
# Step 5e — Set your OpenAI API Key
import openai

openai.api_key = "sk-..."  # ← Replace with your actual OpenAI API key

print("OpenAI client ready!")

### Step 5f: Full RAG + Rerank Pipeline

In [ ]:
# Step 5f — Complete pipeline: Retrieve (top-30) → Rerank → Generate Answer

def generate_answer(question, retrieve_k=30, rerank_top_n=5):
    """
    Full RAG + Rerank pipeline:
    1. Retrieve top-30 candidates from ChromaDB (bi-encoder)
    2. Rerank using cross-encoder → keep top-5
    3. Build prompt with top-5 reranked chunks
    4. Generate answer with OpenAI GPT
    """

    # Stage 1: Retrieve
    candidates = retrieve_candidates(question, top_k=retrieve_k)

    # Stage 2: Rerank
    top_chunks, top_scores = rerank_chunks(question, candidates, top_n=rerank_top_n)

    # Stage 3: Build prompt
    prompt = build_prompt(top_chunks, question)

    # Stage 4: Generate answer
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.3
    )

    answer = response.choices[0].message.content
    return answer, top_scores


# Test the full pipeline
answer, scores = generate_answer("What is photosynthesis?")
print(f"Reranker scores used: {scores}\n")
print(f"Answer:\n{answer}")

### Step 5g: Interactive Chatbot Loop

In [ ]:
# Step 5g — Chat with your textbooks (Retrieve 30 → Rerank → Top 5 → Answer)

print("=" * 60)
print("  Educational Chatbot — RAG + Reranking")
print(f"  Subjects: {list(educational_documents.keys())}")
print("  Pipeline: Retrieve top-30 → Rerank → Use top-5 → Answer")
print("  Type 'quit' to exit")
print("=" * 60)

while True:
    question = input("\nYou: ").strip()

    if question.lower() in ["quit", "exit", "q"]:
        print("Goodbye! Happy studying!")
        break

    if not question:
        continue

    print("Chatbot: Retrieving and reranking from your textbooks...")
    answer, scores = generate_answer(question, retrieve_k=30, rerank_top_n=5)
    print(f"[Reranker scores: {scores}]")
    print(f"\nChatbot: {answer}")
    print("-" * 60)